# Credit Card Fraud Detection Modelling.

## Load the train and test data sets.

In [1]:
import os
import pandas as pd

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/AnomalyDetection/CreditCardFraudDetection/"

initial_train_df = pd.read_csv(DATASETS_PATH + "fraudTrain.csv")

initial_test_df = pd.read_csv(DATASETS_PATH + "fraudTest.csv")

initial_train_df["trans_date_trans_time"] = pd.to_datetime(initial_train_df["trans_date_trans_time"])
initial_test_df["trans_date_trans_time"] = pd.to_datetime(initial_test_df["trans_date_trans_time"])

### As can be seen from the EDA, the test data set is a temporal continuation of the train data set, so the time-series features should be calculated on a merged dataset, or necessery values should be taken from the train dataset for test dataset features.

### We are going to to build up features that describe "normal" behaviour of a customer. For that we are going to take temporal and spacial distributions and calculate Z-scores for them. By Z-Score a model is able to detect irregularities (deviation of a value from the normal distribution).

In [2]:
initial_train_df['minutes_of_day'] = initial_train_df['trans_date_trans_time'].dt.hour * 60 + initial_train_df['trans_date_trans_time'].dt.minute
initial_test_df['minutes_of_day'] = initial_test_df['trans_date_trans_time'].dt.hour * 60 + initial_test_df['trans_date_trans_time'].dt.minute

### Calculate a map of average number of minutes of the day for a purchase and its deviation for each credit card holder, using the train dataset.

In [3]:
df_temporal_stats = initial_train_df.groupby('cc_num')['minutes_of_day'].agg(['mean', 'std']).reset_index()
df_temporal_stats.columns = ['cc_num', 'mean_minutes_of_day', 'std_minutes_of_day']

initial_train_df = initial_train_df.merge(df_temporal_stats, on='cc_num', how='left')
initial_test_df = initial_test_df.merge(df_temporal_stats, on='cc_num', how='left')

initial_train_df['temporal_z_score'] = (initial_train_df['minutes_of_day'] - initial_train_df['mean_minutes_of_day']) / initial_train_df['std_minutes_of_day']
initial_test_df['temporal_z_score'] = (initial_test_df['minutes_of_day'] - initial_test_df['mean_minutes_of_day']) / initial_test_df['std_minutes_of_day']


In [4]:
print(initial_train_df)

         Unnamed: 0 trans_date_trans_time               cc_num  \
0                 0   2019-01-01 00:00:18     2703186189652095   
1                 1   2019-01-01 00:00:44         630423337322   
2                 2   2019-01-01 00:00:51       38859492057661   
3                 3   2019-01-01 00:01:16     3534093764340240   
4                 4   2019-01-01 00:03:06      375534208663984   
...             ...                   ...                  ...   
1296670     1296670   2020-06-21 12:12:08       30263540414123   
1296671     1296671   2020-06-21 12:12:19     6011149206456997   
1296672     1296672   2020-06-21 12:12:32     3514865930894695   
1296673     1296673   2020-06-21 12:13:36     2720012583106919   
1296674     1296674   2020-06-21 12:13:37  4292902571056973207   

                                    merchant       category     amt  \
0                 fraud_Rippin, Kub and Mann       misc_net    4.97   
1            fraud_Heller, Gutmann and Zieme    grocery_pos  107.